<a href="https://colab.research.google.com/github/rwcitek/FHIR-to-Parquet/blob/main/fhir-1ksample-duckdb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FHIR data to DuckDB database

From https://www.kaggle.com/code/drscarlat/fhir-starter-parse-healthcare-bundles-into-tables/notebook


## Setup


In [ ]:
%%capture output
%%bash
pip install -U kagglehub duckdb


In [ ]:
import duckdb
import os
import json
import yaml
import kagglehub
import pprint
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
path = kagglehub.dataset_download("drscarlat/fhir-1ksample")
path


## Find JSON files

In [ ]:
!find {path} | sort | head -3


In [ ]:
!find {path} -type f -name '*.json' | sort | head -3


## Pick one FHIR file


In [ ]:
json1 = !find {path} -type f -name '*.json' | sort
len(json1), json1[:10]


### Explore FHIR using jq

In [ ]:
!cat {json1[0]} | jq 'keys' | head


In [ ]:
!cat {json1[0]} | jq 'values' | head


In [ ]:
!cat {json1[0]} | jq 'map_values(type)'


In [ ]:
!cat {json1[0]} | jq '.entry | length'


In [ ]:
!cat {json1[0]} | jq '.entry[0] | keys' | head


In [ ]:
!cat {json1[0]} | jq -r '.entry[].resource.resourceType' | sort | uniq -c | sort -rn


### Explore one FHIR file using Python

In [ ]:
with open(json1[0], 'r') as f:
  fhir_data = json.load(f)


In [ ]:
# top-level key values
for k ,v in fhir_data.items():
  if not isinstance(v, (list, dict)):
    print(f"{k=}, type={type(v)}, {v=}")
  else:
    print(f"{k=}, type={type(v)}")


In [ ]:
# Explore using pretty print
pprint.pprint(fhir_data, depth=4)

For the single bundle, get count of each resourceType

In [ ]:
pd.Series(
  v["resource"]["resourceType"]
    for i, v in enumerate(fhir_data["entry"])
).value_counts()

## Using DuckDB, scan each FHIR file to determine resource type of the files


In [ ]:
# point to the JSON files in the kagglehub path
json_files = os.path.join(path, "**/*.json")
json_files


In [ ]:
# This runs the scan and returns a Pandas-like result
query = f'''
  SELECT
    resourceType,
    type,
    count(1) as count
  FROM
    read_json_auto(
      '{json_files}',
      maximum_object_size=50331648
    )
  GROUP BY ALL
'''

print(query)


In [ ]:
duckdb.query(query).df()


### Drill down: count number of events per bundle

In [ ]:
query = f'''
  SELECT
    reverse(string_split(reverse(filename), '/')[1]) AS bundle_id,
    resourceType,
    len(entry) AS total_entry_count
  FROM
    read_json_auto(
      '{json_files}',
      maximum_object_size=50331648,
      filename=true
    )
  ORDER BY total_entry_count DESC
'''

print(query)

In [ ]:
df = duckdb.query(query).df()
df

#### Plot


In [ ]:

df["short"] = df["bundle_id"].str[:10]

ax = df.sort_values( by = ["total_entry_count"])[-20:].plot(
  figsize=(12,6),
  kind = 'barh',
  x='short',
  y='total_entry_count',
  ylabel = "bundle ID",
  title="FHIR Resource Distribution"
)

plt.show()


## Read all JSON into a parquet file

Have duckdb file for storing views


In [ ]:
# Create duckdb file
con = duckdb.connect('fhir_data.db')


In [ ]:
!ls -l


### Create bundle parquet file from FHIR/JSON files with all bundles and a key

In [ ]:
parquet_file = 'fhir_bundles.parquet'

con.execute(f'''
COPY (
  SELECT
    reverse(string_split(reverse(filename), '/')[1]) AS bundle_id,
    json AS bundle_data
  FROM
    read_json_objects(
      '{json_files}',
      maximum_object_size=50331648,
      filename=true
    )
) TO '{parquet_file}' (FORMAT PARQUET) ;
'''
) ;


In [ ]:
!ls -l --si fhi*

In [ ]:
!du --si -s {json_files.rsplit("/",2)[0]}

Definitely a reduction in size.


In [ ]:
con.query(f"""
  describe {parquet_file}
""").df()


### Create bundle view

This simplifies queries.

In [ ]:
con.execute(f'''
  CREATE OR REPLACE VIEW bundles AS
    SELECT
      string_split(string_split(bundle_id, '_')[3], '.')[1] as bundle_id,
      bundle_data
    FROM '{parquet_file}'
'''
) ;


In [ ]:
!ls -l --si fhir*


In [ ]:
con.query("""
  describe bundles
""").df()


In [ ]:
con.query("""
  SELECT
    bundle_id,
    bundle_data
  FROM bundles
  LIMIT 1
""").df()


### Drill down one level into JSON


In [ ]:
con.query("""
  SELECT
    bundle_id,
    bundle_data->>'$.resourceType' as res_type,
    bundle_data->>'$.type' as bundle_type
  FROM bundles
  LIMIT 10
""").df()


### Drill down more than one level of the JSON into the Events

In [ ]:
con.query("""
  SELECT
    bundle_id,
    -- 1. Unnest the entry list
    -- 2. Access the 'resource' key
    -- 3. Extract the 'resourceType' ... for each entry in the list
    unnest((bundle_data->'$.entry')::JSON[]).resource.resourceType AS event_type
  FROM bundles
""").df()


### Explode both resource and resourceType using CTE

In [ ]:
con.execute('''
CREATE OR REPLACE VIEW events AS
  WITH
    events AS (
      SELECT
        bundle_id,
        unnest((bundle_data->'$.entry')::JSON[]) as e
      FROM bundles
  ),
    resources as (
      SELECT
        bundle_id,
        e->'$.resource' as res_json
      FROM events
  )

  SELECT
    bundle_id,
    res_json->>'$.resourceType' as res_type,
    res_json
  FROM resources
''')


In [ ]:
con.query('''
  select *
  from events
  limit 10
''').df()



### Total counts for each resource

In [ ]:
df = con.query('''
  SELECT res_type, count(1) as count
  FROM events
  GROUP BY 1
  ORDER BY count DESC
''').df()
df


#### Plot

In [ ]:
df.sort_values( by = ["count"] ).plot(
  figsize=(12,6),
  kind = 'barh',
  x='res_type',
  y='count',
  ylabel = "Resource",
  title="FHIR Resource Distribution",
)

plt.show()


In [ ]:
!ls -l --si

In [ ]:
con.query(f'''
  select count(1)
  from events
'''
)

## PATIENT


### Create view

In [ ]:
# PATIENT = pd.DataFrame(columns=['PatientUID', 'NameFamily', 'NameGiven', 'DoB', 'Gender'])

# print(onePatientID)
# print(onePatient.name[0].family)
# print(onePatient.name[0].given[0])
# print(onePatient.birthDate)
# print(onePatient.gender)

con.execute("""
CREATE OR REPLACE VIEW patient AS
  SELECT
    bundle_id,
    res_json->>'$.resourceType' as res_type,

    res_json->>'$.id' AS PatientUID,
    res_json->>'$.name[0].family' AS NameFamily,
    res_json->>'$.name[0].given[0]' as NameGiven,
    res_json->>'$.birthDate' AS DoB,
    res_json->>'$.gender' AS Gender,

    res_json
  FROM events
  WHERE res_type = 'Patient'
""")


### Query view

In [ ]:
con.query("""
  select count(1) as count
  from patient
"""
).df()


In [ ]:
df = con.query("""
  select *
  from patient
"""
).df()
df


In [ ]:
pprint.pprint(json.loads(df["res_json"][0]))

### Create parquet file

In [ ]:
parquet_file = 'fhir_patient.parquet'

con.execute(f'''
COPY (
  SELECT *
  FROM patient
) TO '{parquet_file}' (FORMAT PARQUET) ;
'''
)


### Query parquet file

In [ ]:
!ls -la {parquet_file}


In [ ]:
con.query(f"""
  select count(1) as count
  from '{parquet_file}'
"""
).df()


## CONDITION


### Create view

In [ ]:
# CONDITION = pd.DataFrame(columns=['ConditionText', 'ConditionOnsetDates', 'PatientUID'])

con.execute(f"""
CREATE OR REPLACE VIEW condition AS
  SELECT
    bundle_id,
    res_json->>'$.resourceType' as res_type,

    res_json->>'$.code.text' as ConditionText,
    res_json->>'$.onsetDateTime' as ConditionOnsetDates,
    reverse(string_split(reverse(res_json->>'$.subject.reference'), ':')[1]) AS PatientUID,

    res_json
  FROM events
  WHERE res_type = 'Condition'
""")


### Query view

In [ ]:
con.query("""
  select count(1) as count
  from condition
"""
).df()


In [ ]:
df = con.query("""
  select *
  from condition
"""
).df()
df


In [ ]:
pprint.pprint(json.loads(df["res_json"][0]))

### Create parquet file

In [ ]:
parquet_file = 'fhir_condition.parquet'

con.execute(f'''
COPY (
  SELECT *
  FROM condition
) TO '{parquet_file}' (FORMAT PARQUET) ;
'''
)


### Query parquet file

In [ ]:
!ls -la {parquet_file}


In [ ]:
con.query(f"""
  select count(1) as count
  from '{parquet_file}'
"""
).df()


## OBSERVATION

### Create View

In [ ]:
# OBSERVATION = pd.DataFrame(columns=['ObservationText', 'ObservationValue', 'ObservationUnit', 'ObservationDate', 'PatientUID'])

con.execute("""
CREATE OR REPLACE VIEW observation AS
  SELECT
    bundle_id,
    res_json->>'$.resourceType' as res_type,

    res_json->>'$.code.text' AS ObservationText,
    res_json->>'$.valueQuantity.value' AS ObservationValue,
    res_json->>'$.valueQuantity.unit' AS ObservationUnit,
    res_json->>'$.issued' AS ObservationDate,

    reverse(string_split(reverse(res_json->>'$.subject.reference'), ':')[1]) AS PatientUID,
    res_json
  FROM events
  WHERE res_type = 'Observation'
""")



### Query view


In [ ]:
con.query("""
  select count(1) as count
  from observation
"""
).df()


In [ ]:
df = con.query("""
  select *
  from observation
"""
).df()
df


In [ ]:
pprint.pprint(json.loads(df["res_json"][0]))

### Create parquet file

In [ ]:
parquet_file = 'fhir_observation.parquet'

con.execute(f'''
COPY (
  SELECT *
  FROM observation
) TO '{parquet_file}' (FORMAT PARQUET) ;
'''
)


### Query parquet file

In [ ]:
!ls -la {parquet_file}


In [ ]:
con.query(f"""
  select count(1) as count
  from '{parquet_file}'
"""
).df()


# ==== START HERE =======


In [ ]:
stop

In [ ]:


# MEDICATION = pd.DataFrame(columns=['MedicationText', 'MedicationDates', 'PatientUID'])
# PROCEDURE = pd.DataFrame(columns=['ProcedureText', 'ProcedureDates', 'PatientUID'])
# ENCOUNTER = pd.DataFrame(columns=['EncountersText', 'EncounterLocation', 'EncounterProvider','EncounterDates', 'PatientUID'])
# CLAIM = pd.DataFrame(columns=['ClaimProvider', 'ClaimInsurance', 'ClaimDate', 'ClaimType','ClaimItem','ClaimUSD', 'PatientUID'])
# IMMUNIZATION = pd.DataFrame(columns=['Immunization', 'ImmunizationDates', 'PatientUID'])



# == Demo


In [ ]:
duckdb.query('''
describe  'https://ddc-datascience.s3.amazonaws.com/big-data/fhir_bundles.parquet'
'''
)

In [ ]:
duckdb.query('''
select count(1) as count
from  'https://ddc-datascience.s3.amazonaws.com/big-data/fhir_bundles.parquet'
'''
)

In [ ]:
duckdb.query('''
select bundle_id,
    bundle_data->>'$.resourceType' as res_type,
    bundle_data->>'$.type' as bundle_type
from  'https://ddc-datascience.s3.amazonaws.com/big-data/fhir_bundles.parquet'
limit 20
  ;
'''
)

What we see is that each JSON file is a FHIR "Bundle".  A Bundle is a collection of FHIR resources that need to be handled together.  In this case

## Create DuckDB database of FHIR data

In [ ]:
# setup DuckDB
con = duckdb.connect('fhir_data.duckdb')


In [ ]:
# The 'maximum_depth' is helpful for complex FHIR structures
con.execute(f"""
    CREATE OR REPLACE TABLE fhir_raw AS
    SELECT * FROM read_json(
      '{json_files}',
      format='auto',
      maximum_depth=3,
      maximum_object_size=50331648
    );
""")


In [ ]:
df_check = con.execute("SELECT resourceType, count(*) as count FROM fhir_raw GROUP BY 1").df()
print(df_check)


In [ ]:
# PATIENT = pd.DataFrame(columns=['PatientUID', 'NameFamily', 'NameGiven', 'DoB', 'Gender'])

con.execute("""
    CREATE OR REPLACE TABLE patients AS
    SELECT
        unnest(entry).resource.id AS patient_id,
        unnest(entry).resource.gender AS gender,
        unnest(entry).resource.birthDate AS birth_date,
        unnest(entry).resource.name[1].family AS last_name
    FROM fhir_raw
    WHERE resourceType = 'Bundle';
""")


In [ ]:
con.execute("SELECT * FROM patients LIMIT 5").df()


In [ ]:
con.query("SELECT * FROM fhir_raw limit 1")


In [ ]:
raw_json = con.execute("SELECT entry->'$[0]' FROM fhir_raw LIMIT 1").fetchone()[0]


In [ ]:
len(raw_json)


In [ ]:
f"{raw_json[:100]} ... {raw_json[-100:]}"


In [ ]:
parsed_json = json.loads(raw_json)
print(json.dumps(parsed_json, indent=2))


In [ ]:
con.query("""
  with foo as (
    SELECT
        -- 1. Unnest the array into individual resource objects
        unnest(entry).resource.resourceType AS resource_type1,
        unnest(entry).resource.id AS patient_id1,
        -- unnest(entry).resource as resource1,

        unnest(entry)->>'$.resource.resourceType' AS resource_type,
        unnest(entry)->>'$.resource.id' AS resource_id,
        -- unnest(entry)->'$.resource' as resource
    FROM fhir_raw
  )
  select * from foo
  where
    -- resource_id like '225d%' and
    resource_type = 'Patient'
  limit 10
""").df().columns

In [ ]:
raw_json = con.execute('''
  with foo as (
    SELECT
        -- 1. Unnest the array into individual resource objects
        unnest(entry)->>'$.resource.resourceType' AS resource_type,
        unnest(entry)->>'$.resource.id' AS resource_id,
        unnest(entry)->>'$.resource' as resource
    FROM fhir_raw
  )
  select resource from foo
  where resource_id like '225d%' and
  resource_type = 'Patient'
  limit 1
'''
).fetchone()[0]


In [ ]:
(
  type(raw_json),
  len(raw_json),
  f"{raw_json[:30]} .. {raw_json[-30:]}"
)

In [ ]:
data_dict = json.loads(raw_json)
yaml_output = yaml.dump(data_dict, default_flow_style=False, sort_keys=True)
print(yaml_output)


In [ ]:
con.query("""
  CREATE TYPE gender_enum AS ENUM ('male', 'female');

  with foo as (
    SELECT
        -- 1. Unnest the array into individual resource objects
        unnest(entry)->>'$.resource.resourceType' AS resource_type,
        unnest(entry)->>'$.resource.id' AS resource_id,
        unnest(entry)->'$.resource' as resource
    FROM fhir_raw
  ),
  patient as (
    select
      resource_id::UUID as PatientUID,
      resource->>'$.name[0].family' as NameFamily,
      resource->>'$.name[0].given[0]' as NameGiven,
      (resource->>'$.birthDate')::DATE as DoB,
      (resource->>'$.gender')::gender_enum as Gender
    from foo
    where resource_type = 'Patient'
  )
  select *
  from patient
  limit 10
""")
